# end-grad-default-ones-like — ex2: resolve end_grad with explicit per-sample weights on a (B,) loss

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `end-grad-default-ones-like`. Running the final beacon cell reports progress against the `Backprop: end-grad ones_like default` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: end-grad ones_like default` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`end-grad-default-ones-like`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "end-grad-default-ones-like"
DD_SUBTOPIC = "Backprop: end-grad ones_like default"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Non-scalar end_grad with explicit shape — quick refresher

ex1 covered the `end_grad=None → ones_like` default. The deeper facet: when the user passes `end_grad` EXPLICITLY, every per-position value is a weighting on the seed gradient.

```python
loss = per_sample_loss(x, y)   # shape (B,)
weights = (y == HARD_CLASS).float()  # 1 for hard samples, 0 otherwise
loss.backward(weights)         # only hard samples contribute
```

Inside `resolve_end_grad`, when `end_grad` is supplied, it must:
1. Be a `MiniTensor` (not a raw tensor — the public API is consistent).
2. Match `end_node.array.shape` element-for-element.
3. Be unboxed to its `.array` (the rest of `backprop` works on raw tensors).

### Exercise 2 — resolve end_grad with explicit per-sample weights on a (B,) loss

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply the explicit end_grad path: validate the supplied MiniTensor's shape against the end node, unbox it, and demonstrate that per-sample weighting flows through to per-leaf grads as a linear scale.
> Keywords: end-grad, per-sample-weights, loss-vector, unbox
> ```

**KCs targeted:** `end-grad-default-ones-like`, `buffer-copy-inplace`

Implement `weighted_seed_and_chain(end_node, end_grad, x_leaf)` — a tiny demonstration that the explicit `end_grad` is honored ELEMENTWISE in the reverse pass.

Setup (we build this inside the test):
- `x_leaf`: a `(B,)` MiniTensor leaf, `requires_grad=True`.
- `end_node`: a `(B,)` MiniTensor where `end_node = log(x_leaf)` (we build the Recipe by hand).
- `end_grad`: a `(B,)` MiniTensor of per-sample weights.

Your function must:

1. **Resolve the seed.** If `end_grad is None`, default to `t.ones_like(end_node.array)`. Otherwise unbox `end_grad.array` after asserting `end_grad.array.shape == end_node.array.shape` (raise `AssertionError` with a helpful message).
2. **Apply one step of the reverse chain.** `end_node`'s recipe is `log`, so `dL/dx_leaf = seed * (1 / x_leaf.array)`.
3. **Return `(seed, dL_dx_leaf)`** — both raw `torch.Tensor`.

The test verifies:
- shape and dtype propagation,
- explicit weights produce per-position scaling on the leaf grad,
- `end_grad=None` reproduces `ones_like` behavior,
- shape-mismatch raises with a useful message,
- per-sample weighting matches torch.autograd on the equivalent weighted-sum loss `(weights * log(x)).sum()`.

Do NOT call `torch.autograd`.

In [ ]:
def weighted_seed_and_chain(end_node, end_grad, x_leaf):
    """Resolve seed, then chain one log_back step. Returns (seed, dL_dx)."""
    raise NotImplementedError()


def _test_ex2():
    # --- build x_leaf and end_node = log(x_leaf) by hand ---
    x_raw = t.tensor([2.0, 4.0, 8.0, 16.0])
    x_leaf = MiniTensor(x_raw, requires_grad=True)
    end_node = MiniTensor(t.log(x_raw), requires_grad=True)
    end_node.recipe = Recipe(func=t.log, args=(x_raw,), kwargs={}, parents={0: x_leaf})

    # --- explicit per-sample weights: emphasize hard samples (indices 2, 3) ---
    weights = MiniTensor(t.tensor([0.0, 0.0, 1.0, 1.0]))
    seed, dL_dx = weighted_seed_and_chain(end_node, weights, x_leaf)

    # --- seed is the unboxed weights ---
    assert isinstance(seed, t.Tensor) and not isinstance(seed, MiniTensor)
    assert t.allclose(seed, weights.array), f'seed should unbox weights: {seed}'
    assert seed is weights.array, 'unbox should be identity, not copy'

    # --- dL/dx_leaf = weights / x_raw, position-wise ---
    expected = t.tensor([0.0, 0.0, 1.0/8.0, 1.0/16.0])
    assert t.allclose(dL_dx, expected, atol=1e-7), f'dL/dx wrong: {dL_dx} vs {expected}'
    assert dL_dx.shape == x_raw.shape, f'leaf grad shape: {dL_dx.shape}'

    # --- end_grad=None falls back to ones_like ---
    seed_none, dL_dx_none = weighted_seed_and_chain(end_node, None, x_leaf)
    assert t.allclose(seed_none, t.ones(4)), f'None → ones_like: {seed_none}'
    assert t.allclose(dL_dx_none, 1.0 / x_raw, atol=1e-7), f'None chain: {dL_dx_none}'

    # --- dtype preserved by ones_like ---
    fp64_raw = t.tensor([1.0, 2.0], dtype=t.float64)
    fp64_leaf = MiniTensor(fp64_raw, requires_grad=True)
    fp64_end = MiniTensor(t.log(fp64_raw), requires_grad=True)
    fp64_end.recipe = Recipe(func=t.log, args=(fp64_raw,), kwargs={}, parents={0: fp64_leaf})
    seed64, _ = weighted_seed_and_chain(fp64_end, None, fp64_leaf)
    assert seed64.dtype == t.float64, f'dtype preserved: {seed64.dtype}'

    # --- shape mismatch raises ---
    wrong = MiniTensor(t.zeros(2, 3))
    try:
        weighted_seed_and_chain(end_node, wrong, x_leaf)
    except AssertionError as e:
        msg = str(e)
        assert '(2, 3)' in msg or '2, 3' in msg, f'msg missing end_grad shape: {msg!r}'
        assert '(4,)' in msg or '4,' in msg, f'msg missing end_node shape: {msg!r}'
    else:
        raise AssertionError('shape mismatch should have raised')

    # --- per-sample weighting matches torch.autograd on weighted-sum loss ---
    x_ref = x_raw.clone().requires_grad_(True)
    w_ref = weights.array.clone()
    loss = (w_ref * t.log(x_ref)).sum()
    loss.backward()
    assert t.allclose(dL_dx, x_ref.grad, atol=1e-7), (
        f'weighted chain disagrees with autograd: ours={dL_dx}, ref={x_ref.grad}'
    )
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def weighted_seed_and_chain(end_node, end_grad, x_leaf):
    # 1. resolve the seed.
    if end_grad is None:
        seed = t.ones_like(end_node.array)
    else:
        assert end_grad.array.shape == end_node.array.shape, (
            f'end_grad shape {tuple(end_grad.array.shape)} mismatches '
            f'end_node shape {tuple(end_node.array.shape)}'
        )
        seed = end_grad.array
    # 2. one step of log_back: dL/dx = seed / x.
    dL_dx = seed / x_leaf.array
    return seed, dL_dx
```

**Why explicit weights are equivalent to `(weights * loss).sum()`.** Passing `end_grad=w` to `loss.backward()` is mathematically the same as calling `(w * loss).sum().backward()`. The chain rule's seed multiplies through every downstream back fn — putting it in the seed vs in a wrapping `.sum()` produces identical leaf grads.

**Why unbox is identity, not copy.** `end_grad.array` is the raw tensor; we want to use it directly as the seed for the rest of backprop. Cloning would double memory for no reason. The `is` test in the harness pins this down — implementations that `return end_grad.array.clone()` would fail it.

**Why ones_like preserves dtype.** `t.ones_like(x)` reads `x.dtype` by default. If you write `t.ones(end_node.array.shape)` you get `float32` regardless — silently demoting `float64` end nodes.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()